# CoffeeLeafVision — Training (Kaggle GPU)

Entrena las 4 arquitecturas (ResNet50, EfficientNetV2-S, MobileNetV3-Large, ViT-Small) sobre `BRACOL + RoCoLe` con 5-fold stratified CV.

**Cómo correr en Kaggle:**
1. Subir el repo a Kaggle como dataset privado o clonar desde GitHub.
2. Subir BRACOL y RoCoLe como datasets de Kaggle, adjuntarlos al notebook.
3. Accelerator: `GPU T4 x1` o `P100`.
4. Run All. Wall clock: 3-4 horas.

Al final los checkpoints quedan en `/kaggle/working/checkpoints/`. Súbelos a HF Hub con `scripts/upload_to_hf.py`.

In [ ]:
import os
import sys
from pathlib import Path

sys.path.insert(0, "/kaggle/working/coffee-leaf-vision")

import torch
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("Device:", torch.cuda.get_device_name(0))

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [ ]:
from pathlib import Path
from src.preprocess import build_manifest, stratified_split

BRACOL = Path("/kaggle/input/bracol-coffee-dataset/BRACOL")
ROCOLE = Path("/kaggle/input/rocole-coffee-dataset/RoCoLe")

manifest = build_manifest(BRACOL, ROCOLE)
splits = stratified_split(manifest, test_size=0.15, n_folds=5, random_state=42)
print(f"Total: {len(manifest)}, test: {len(splits['test_idx'])}, folds: {len(splits['folds'])}")

In [ ]:
import json
from pathlib import Path

import torch
from torch.utils.data import DataLoader

from src.data import CoffeeLeafDataset, build_train_transform, build_val_transform
from src.models import build_model
from src.train import train_fold
from src.evaluate import aggregate_fold_metrics, count_parameters, model_size_mb, measure_inference_latency

CKPT_DIR = Path("/kaggle/working/checkpoints")
CKPT_DIR.mkdir(parents=True, exist_ok=True)

ARCHITECTURES = ["resnet50", "efficientnetv2_s", "mobilenetv3_large", "vit_small"]
BATCH_SIZE = 32

all_results = {}

for arch in ARCHITECTURES:
    print(f"\n=== {arch} ===")
    fold_metrics = []
    for fold_idx, (train_idx, val_idx) in enumerate(splits["folds"]):
        print(f"  fold {fold_idx + 1}/5")
        train_loader = DataLoader(
            CoffeeLeafDataset(manifest.loc[train_idx], build_train_transform()),
            batch_size=BATCH_SIZE, shuffle=True, num_workers=2,
        )
        val_loader = DataLoader(
            CoffeeLeafDataset(manifest.loc[val_idx], build_val_transform()),
            batch_size=BATCH_SIZE, shuffle=False, num_workers=2,
        )
        model = build_model(arch, num_classes=5, pretrained=True).to(DEVICE)
        ckpt_path = CKPT_DIR / f"{arch}_fold{fold_idx}.pt"
        result = train_fold(
            model, train_loader, val_loader, DEVICE,
            warmup_epochs=10, finetune_epochs=20,
            checkpoint_path=ckpt_path,
        )
        fold_metrics.append(result["best_metrics"])

    cv_summary = aggregate_fold_metrics(fold_metrics)

    sample_model = build_model(arch, num_classes=5, pretrained=False)
    all_results[arch] = {
        "cv_summary": cv_summary,
        "fold_metrics": fold_metrics,
        "n_params": count_parameters(sample_model),
        "size_mb": model_size_mb(sample_model),
        "latency_cpu": measure_inference_latency(sample_model, device="cpu", n_runs=20),
    }
    del sample_model

(CKPT_DIR / "results.json").write_text(json.dumps(all_results, indent=2))
print("\nDone. Results saved to /kaggle/working/checkpoints/results.json")

## Después de entrenar

1. Identificar arquitectura ganadora por `cv_summary["f1_macro"]["mean"]`.
2. Evaluar contra el hold-out (notebook `03_evaluation.ipynb`).
3. Subir `results.json` + el mejor checkpoint a HF Hub con `scripts/upload_to_hf.py`.